In [2]:
import cv2
import numpy as np
import joblib
import mediapipe as mp

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

In [3]:
import sys
import os

sys.path.append(os.path.abspath('..'))
from utils.features import extract_features

In [4]:
model_knn = joblib.load('../models/model_knn.pkl')
model_svm = joblib.load('../models/model_svm.pkl')
scaler = joblib.load('../models/scaler.pkl')

In [5]:
options = HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(
        model_asset_path='../hand_landmarker.task'
    ),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

landmarker = HandLandmarker.create_from_options(options)

- KNN

In [6]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    
    result = landmarker.detect(mp_image)
    
    if result.hand_landmarks and len(result.hand_landmarks)>0:
        hand = result.hand_landmarks[0]
        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand])
        
        features = extract_features(coords)
        features = scaler.transform(features.reshape(1, -1))
        pred = model_knn.predict(features)[0]
        
        cv2.putText(frame, f'Pred: {pred}', (10,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        
    cv2.imshow("Realtime Sign Language", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

- SVM

In [7]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    
    result = landmarker.detect(mp_image)
    
    if result.hand_landmarks and len(result.hand_landmarks)>0:
        hand = result.hand_landmarks[0]
        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand])
        
        features = extract_features(coords)
        features = scaler.transform(features.reshape(1, -1))
        pred = model_knn.predict(features)[0]
        
        cv2.putText(frame, f'Pred: {pred}', (10,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        
    cv2.imshow("Realtime Sign Language", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()